### Main Code

In [ ]:
## v1, withoput bias
'''
import numpy as np

def hex_to_signed16(h):
    """Convert hex string (no prefix) to signed 16-bit int."""
    val = int(h, 16)
    return val if val < 0x8000 else val - 0x10000
def unsigned8_to9(x):
    """Unsign extend 8b to 9b in Python."""
    return x & 0x1FF
def signed8_to9(x):
    """Sign extend 8b to 9b in Python."""
    return x if x < 0x80 else x - 0x100

def signed_16x16_by_9x9(a, b, verbose=True):
    """Perform signed 16x16 = 32b multiplication using four 9x9 multipliers."""
    a = np.int16(a)
    b = np.int16(b)
    a_lo = unsigned8_to9(a & 0xFF)
    a_hi = signed8_to9((a >> 8) & 0xFF)
    b_lo = unsigned8_to9(b & 0xFF)
    b_hi = signed8_to9((b >> 8) & 0xFF)

    p0 = a_lo * b_lo
    p1 = a_lo * b_hi
    p2 = a_hi * b_lo
    p3 = a_hi * b_hi

    res = p0 + ((p1 + p2) << 8) + (p3 << 16)

    if verbose:
        print(f"🔹 Multiply {a:6d} (0x{a & 0xFFFF:04x}) * {b:6d} (0x{b & 0xFFFF:04x})")
        print(f"    a_hi={a_hi:4d} (0x{a_hi & 0x1FF:03x}), a_lo={a_lo:4d} (0x{a_lo & 0x1FF:03x})")
        print(f"    b_hi={b_hi:4d} (0x{b_hi & 0x1FF:03x}), b_lo={b_lo:4d} (0x{b_lo & 0x1FF:03x})")
        print(f"    p0 = {p0:8d} (0x{p0 & 0xFFFFFFFF:08x})")
        print(f"    p1 = {p1:8d} (0x{p1 & 0xFFFFFFFF:08x})")
        print(f"    p2 = {p2:8d} (0x{p2 & 0xFFFFFFFF:08x})")
        print(f"    p3 = {p3:8d} (0x{p3 & 0xFFFFFFFF:08x})")
        print(f"    res = {res:11d} (0x{res & 0xFFFFFFFF:08x})\n")
              
    return res

def conv1d_16x16_using_9x9(ifmap_hex, kernel_hex, stride=1, pad=0):
    ch_in = len(ifmap_hex)
    ifm_len = len(ifmap_hex[0])
    ker_len = len(kernel_hex[0][0])
    ch_out = len(kernel_hex)

    # Decode input
    ifmap = np.array([[hex_to_signed16(val) for val in row] for row in ifmap_hex], dtype=np.int16)
    kernel = np.array([[[hex_to_signed16(val) for val in ker_ic] for ker_ic in ker_oc] for ker_oc in kernel_hex], dtype=np.int16)

    # Padding
    ifmap = np.pad(ifmap, ((0, 0), (pad, pad)), mode="constant", constant_values=0)
    out_len = (ifmap.shape[1] - ker_len) // stride + 1

    # Output buffer
    output = np.zeros((ch_out, out_len), dtype=np.int32)

    # Convolution
    for oc in range(ch_out):
        for i in range(out_len):
            acc = 0
            print(f"\n🟨 Output Channel {oc}, Position {i}:")
            for ic in range(ch_in):
                window = ifmap[ic, i * stride : i * stride + ker_len]
                kernel_slice = kernel[oc, ic, :]
                for k in range(ker_len):
                    prod = signed_16x16_by_9x9(window[k], kernel_slice[k], verbose=True)
                    acc += prod
            output[oc, i] = acc
            print(f"➡️ Accumulated sum: {acc} ({acc & 0xFFFFFFFF:08x})\n")

    return output
'''

In [35]:
import numpy as np

def hex_to_signed16(h):
    val = int(h, 16)
    return val if val < 0x8000 else val - 0x10000

def hex_to_signed32(h):
    val = int(h, 16)
    return val if val < 0x80000000 else val - 0x100000000

def unsigned8_to9(x):
    return x & 0x1FF

def signed8_to9(x):
    return x if x < 0x80 else x - 0x100


def signed_16x16_by_9x9(a, b, print_mode=0):
    """Perform signed 16x16 = 32b multiplication using four 9x9 multipliers."""
    a = np.int16(a)
    b = np.int16(b)
    a_lo = unsigned8_to9(a & 0xFF)
    a_hi = signed8_to9((a >> 8) & 0xFF)
    b_lo = unsigned8_to9(b & 0xFF)
    b_hi = signed8_to9((b >> 8) & 0xFF)

    p0 = a_lo * b_lo
    p1 = a_lo * b_hi
    p2 = a_hi * b_lo
    p3 = a_hi * b_hi

    tmp0 = p0
    tmp1 = tmp0 + (p1 << 8)
    tmp2 = tmp1 + (p2 << 8)
    res  = tmp2 + (p3 << 16)

    if print_mode == 2:
        print(f"🔹 Multiply {a:6d} (0x{a & 0xFFFF:04x}) * {b:6d} (0x{b & 0xFFFF:04x})")
        print(f"    a_hi={a_hi:4d} (0x{a_hi & 0x1FF:03x}), a_lo={a_lo:4d} (0x{a_lo & 0x1FF:03x})")
        print(f"    b_hi={b_hi:4d} (0x{b_hi & 0x1FF:03x}), b_lo={b_lo:4d} (0x{b_lo & 0x1FF:03x})")
        print(f"    p0 = {p0:8d} (0x{p0 & 0xFFFFFFFF:08x})")
        print(f"    p1 = {(p1 << 8):8d} (0x{(p1 << 8) & 0xFFFFFFFF:08x}), accum = {tmp1:8d} (0x{tmp1 & 0xFFFFFFFF:08x})")
        print(f"    p2 = {(p2 << 8):8d} (0x{(p2 << 8) & 0xFFFFFFFF:08x}), accum = {tmp2:8d} (0x{tmp2 & 0xFFFFFFFF:08x})")
        print(f"    p3 = {(p3 << 16):8d} (0x{(p3 << 16) & 0xFFFFFFFF:08x}), accum = {res:8d} (0x{res & 0xFFFFFFFF:08x})")
        print(f"    res = {res:11d} (0x{res & 0xFFFFFFFF:08x})\n")

    return np.int32(res)


def conv1d_16x16_using_9x9(ifmap_hex, kernel_hex, stride=1, pad=0, bias_hex=None, print_mode=1):
    """
    1D convolution using signed 16x16 -> 32b multiplication decomposed into 9x9 multipliers.

    print_mode:
        0 = no print
        1 = print accumulated sum only
        2 = full verbose mode
    """
    ch_in = len(ifmap_hex)
    ifm_len = len(ifmap_hex[0])
    ker_len = len(kernel_hex[0][0])
    ch_out = len(kernel_hex)

    # Decode input and kernel
    ifmap = np.array([[hex_to_signed16(val) for val in row] for row in ifmap_hex], dtype=np.int16)
    kernel = np.array([[[hex_to_signed16(val) for val in ker_ic] for ker_ic in ker_oc] for ker_oc in kernel_hex], dtype=np.int16)

    # Decode bias (32-bit)
    if bias_hex is not None:
        if len(bias_hex) != ch_out:
            raise ValueError(f"❌ Bias 長度錯誤: 需要 {ch_out} 組，但提供了 {len(bias_hex)}")
        bias = np.array([hex_to_signed32(b) for b in bias_hex], dtype=np.int32)
    else:
        bias = np.zeros(ch_out, dtype=np.int32)

    # Padding
    ifmap = np.pad(ifmap, ((0, 0), (pad, pad)), mode="constant", constant_values=0)
    out_len = (ifmap.shape[1] - ker_len) // stride + 1

    # Output buffer
    output = np.zeros((ch_out, out_len), dtype=np.int32)

    # Convolution
    for oc in range(ch_out):
        for i in range(out_len):
            acc = np.int32(0)
            if print_mode == 2:
                print(f"\n🟨 Output Channel {oc}, Position {i}:")
            for ic in range(ch_in):
                window = ifmap[ic, i * stride : i * stride + ker_len]
                kernel_slice = kernel[oc, ic, :]
                for k in range(ker_len):
                    prod = signed_16x16_by_9x9(window[k], kernel_slice[k], print_mode)
                    acc += prod
            acc += bias[oc]
            output[oc, i] = acc
            if print_mode >= 1:
                print(f"➡️ Accumulated sum (with bias {bias[oc]}): {acc} (0x{acc & 0xFFFFFFFF:08x})")

    return output


In [40]:
## test
ifmap = [
    ["0067", "ffca"],
    ["ff09", "fff4"]
]

kernel = [
    [
        ["ff79", "ff54"],
        ["00aa ", "ffcb"]
    ]
]

# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=2, pad=0, print_mode=1)

➡️ Accumulated sum (with bias 0): -45971 (0xffff4c6d)


In [41]:
## conv1 ofm_0_0
ifmap = [
    ["ffaf", "ffb1", "ffb1"]
]

kernel = [
    [
        ["ffcb", "fff6", "ff5d"]
    ]
]

bias = ["fffff400"]

# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=1, pad=0, bias_hex=bias, print_mode=1)

➡️ Accumulated sum (with bias -3072): 14888 (0x00003a28)


### conv2 ofm0_0

In [65]:
## conv2 ofm_0_0
ifmap = [
    ["003a", "003b", "003d", "003a", "003a", "003c", "003c"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["0059", "005a", "005c", "0059", "0059", "005b", "005b"],
    ["008e", "0090", "0092", "008e", "008e", "0091", "0091"],
    ["009e", "00a0", "00a1", "009e", "009e", "00a1", "00a1"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["0094", "0096", "009d", "009a", "0096", "0097", "009d"],
    ["009e", "00a0", "00a2", "009f", "009e", "00a1", "00a1"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["0000", "0000", "0000", "0000", "0000", "0000", "0000"],
    ["00ca", "00c0", "00c6", "00cf", "00c3", "00c1", "00ca"],
    ["004b", "004b", "004d", "004c", "004a", "004c", "004d"],
    ["0079", "0082", "0081", "0078", "007e", "0083", "007e"]
]

kernel = [
    [
        ["0016", "0042", "0048"],
        ["fff4", "005d", "000b"],
        ["0087", "0006", "ff7f"],
        ["0057", "ffc0", "0008"],
        ["ffbb", "fffe", "003a"],
        ["ffdc", "ffd2", "ffeb"],
        ["ffdf", "ffcc", "fff1"],
        ["ffe8", "005d", "ffe8"],
        ["ffee", "005f", "ffc6"],
        ["ffd2", "0008", "001b"],
        ["ff90", "0039", "ffee"],
        ["0015", "ffa4", "ff69"],
        ["ff97", "000b", "0000"],
        ["0095", "0013", "ff97"],
        ["fff0", "ffc8", "fff0"],
        ["0070", "ffb9", "000f"]
    ]
]

bias = ["00009d00"]
# bias = ["00000000"]
# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=2, pad=0, bias_hex=bias, print_mode=1)

➡️ Accumulated sum (with bias 40192): 56597 (0x0000dd15)
➡️ Accumulated sum (with bias 40192): 58139 (0x0000e31b)
➡️ Accumulated sum (with bias 40192): 55571 (0x0000d913)


In [ ]:
## conv2 ofm_0_1
# pe0
ifmap = [
    ["003d", "003a", "003a"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["005c", "0059", "0059"]
]

kernel = [
    [
        ["0016", "0042", "0048"],
        ["fff4", "005d", "000b"],
        ["0087", "0006", "ff7f"],
        ["0057", "ffc0", "0008"]
    ]
]
# pe0 + pe1 + pe2 + pe3
ifmap = [
    ["003d", "003a", "003a"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["005c", "0059", "0059"],
    ["0092", "008e", "008e"],
    ["00a1", "009e", "009e"],
    ["0000", "0000", "0000"],
    ["009d", "009a", "0096"],
    ["00a2", "009f", "009e"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["00c6", "00cf", "00c3"],
    ["004d", "004c", "004a"],
    ["0081", "0078", "007e"]
]

kernel = [
    [
        ["0016", "0042", "0048"],
        ["fff4", "005d", "000b"],
        ["0087", "0006", "ff7f"],
        ["0057", "ffc0", "0008"],
        ["ffbb", "fffe", "003a"],
        ["ffdc", "ffd2", "ffeb"],
        ["ffdf", "ffcc", "fff1"],
        ["ffe8", "005d", "ffe8"],
        ["ffee", "005f", "ffc6"],
        ["ffd2", "0008", "001b"],
        ["ff90", "0039", "ffee"],
        ["0015", "ffa4", "ff69"],
        ["ff97", "000b", "0000"],
        ["0095", "0013", "ff97"],
        ["fff0", "ffc8", "fff0"],
        ["0070", "ffb9", "000f"]
    ]
]
# bias = ["00009d00"]
bias = ["00000000"]
# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=2, pad=0, bias_hex=bias, print_mode=1)

➡️ Accumulated sum (with bias 0): 17947 (0x0000461b)


### conv2 ofm 0_64


In [79]:
## conv2 ofm_0_0
ifmap = [
    ["000b", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0024", "0011", "0000"],
    ["005a", "0049", "0000"],
    ["006d", "005b", "0000"],
    ["0000", "0000", "0000"],
    ["006c", "0093", "0000"],
    ["0066", "0056", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["000c", "0008", "0000"],
    ["00a9", "00c5", "0000"],
    ["0019", "000f", "0000"],
    ["0043", "0020", "0000"]
]

kernel = [
    [
        ["0016", "0042", "0048"],
        ["fff4", "005d", "000b"],
        ["0087", "0006", "ff7f"],
        ["0057", "ffc0", "0008"],
        ["ffbb", "fffe", "003a"],
        ["ffdc", "ffd2", "ffeb"],
        ["ffdf", "ffcc", "fff1"],
        ["ffe8", "005d", "ffe8"],
        ["ffee", "005f", "ffc6"],
        ["ffd2", "0008", "001b"],
        ["ff90", "0039", "ffee"],
        ["0015", "ffa4", "ff69"],
        ["ff97", "000b", "0000"],
        ["0095", "0013", "ff97"],
        ["fff0", "ffc8", "fff0"],
        ["0070", "ffb9", "000f"]
    ]
]

bias = ["00009d00"]
# bias = ["00000000"]
# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=2, pad=0, bias_hex=bias, print_mode=1)

➡️ Accumulated sum (with bias 40192): 77169 (0x00012d71)


In [89]:
## conv2 ofm_0_0
# pe0 + pe1 + pe3
ifmap = [
    ["000b", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0024", "0011", "0000"],
    ["005a", "0049", "0000"],
    ["006d", "005b", "0000"],
    ["0000", "0000", "0000"],
    ["006c", "0093", "0000"],
    ["0066", "0056", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["0000", "0000", "0000"],
    ["000c", "0008", "0000"],
    ["00a9", "00c5", "0000"],
    ["0019", "000f", "0000"],
    ["0043", "0020", "0000"]
]

kernel = [
    [
        ["0016", "0042", "0048"],
        ["fff4", "005d", "000b"],
        ["0087", "0006", "ff7f"],
        ["0057", "ffc0", "0008"],
        ["ffbb", "fffe", "003a"],
        ["ffdc", "ffd2", "ffeb"],
        ["ffdf", "ffcc", "fff1"],
        ["ffe8", "005d", "ffe8"],
        ["ffee", "005f", "ffc6"],
        ["ffd2", "0008", "001b"],
        ["ff90", "0039", "ffee"],
        ["0015", "ffa4", "ff69"],
        ["ff97", "000b", "0000"],
        ["0095", "0013", "ff97"],
        ["fff0", "ffc8", "fff0"],
        ["0070", "ffb9", "000f"]
    ]
]

bias = ["00000000"]
# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=2, pad=0, bias_hex=bias, print_mode=1)

➡️ Accumulated sum (with bias 0): 36977 (0x00009071)


In [92]:
## conv2 ofm_0_0
# pe4
ifmap = [
    ["000c", "0008", "0000"],
    ["00a9", "00c5", "0000"],
    ["0019", "000f", "0000"],
    ["0043", "0020", "0000"]
]

kernel = [
    [
        ["ff97", "000b", "0000"],
        ["0095", "0013", "ff97"],
        ["fff0", "ffc8", "fff0"],
        ["0070", "ffb9", "000f"]
    ]
]

bias = ["00000000"]
# === 執行卷積 ===
output = conv1d_16x16_using_9x9(ifmap, kernel, stride=2, pad=0, bias_hex=bias, print_mode=2)


🟨 Output Channel 0, Position 0:
🔹 Multiply     12 (0x000c) *   -105 (0xff97)
    a_hi=   0 (0x000), a_lo=  12 (0x00c)
    b_hi=  -1 (0x1ff), b_lo= 151 (0x097)
    p0 =     1812 (0x00000714)
    p1 =    -3072 (0xfffff400), accum =    -1260 (0xfffffb14)
    p2 =        0 (0x00000000), accum =    -1260 (0xfffffb14)
    p3 =        0 (0x00000000), accum =    -1260 (0xfffffb14)
    res =       -1260 (0xfffffb14)

🔹 Multiply      8 (0x0008) *     11 (0x000b)
    a_hi=   0 (0x000), a_lo=   8 (0x008)
    b_hi=   0 (0x000), b_lo=  11 (0x00b)
    p0 =       88 (0x00000058)
    p1 =        0 (0x00000000), accum =       88 (0x00000058)
    p2 =        0 (0x00000000), accum =       88 (0x00000058)
    p3 =        0 (0x00000000), accum =       88 (0x00000058)
    res =          88 (0x00000058)

🔹 Multiply      0 (0x0000) *      0 (0x0000)
    a_hi=   0 (0x000), a_lo=   0 (0x000)
    b_hi=   0 (0x000), b_lo=   0 (0x000)
    p0 =        0 (0x00000000)
    p1 =        0 (0x00000000), accum =        0 